# Benchmark 2 — Fletcher-Reeves vs Polak-Ribière sur fonctions non-convexes

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from src.test_functions import create_system, quadraticn_, multitrous2_, cubic2_, Rosenbrock

In [ ]:
GRID = 50
eps = 1e-6
max_iter = 10000

FUNC_CONFIGS = {
    "multitrous2": {"gammas": [1, 2, 4, 8], "domain": (-4.0, 4.0)},
    "cubic2": {"gammas": [1, 5, 10], "domain": (-4.0, 2.0)},
    "rosenbrock": {"gammas": [1, 10, 100], "domain": (-2.0, 3.0)},
}

OUT = "../figures/benchmark2"
os.makedirs(OUT, exist_ok=True)

plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

In [ ]:
def grad_multitrous2(x, gamma):
    return np.array([
        -40 * x[0] * np.sin(x[0]**2) + 2 * x[0],
        -40 * x[1] * np.sin(x[1]**2) + 2 * gamma * x[1],
    ])


def grad_cubic2(x, gamma):
    return np.array([
        3 * x[0]**2 + 2 * gamma * x[0] + 1,
        3 * x[1]**2 + 2 * gamma * x[1] + 1,
    ])


def grad_rosenbrock(x, gamma):
    return np.array([
        2 * (x[0] - 1) + 4 * gamma * x[0] * (x[0]**2 - x[1]),
        -2 * gamma * (x[0]**2 - x[1]),
    ])

In [ ]:
def _run_one(func_name, gamma, method, i, j, x0_0, x0_1, eps, max_iter):
    np.seterr(all="ignore")

    if func_name == "multitrous2":
        f = lambda x: multitrous2_(x, gamma=gamma)
        gf = lambda x: grad_multitrous2(x, gamma)
    elif func_name == "cubic2":
        f = lambda x: cubic2_(x, gamma=gamma)
        gf = lambda x: grad_cubic2(x, gamma)
    else:
        f = lambda x: Rosenbrock(x, gamma=gamma)
        gf = lambda x: grad_rosenbrock(x, gamma)

    xk = np.array([x0_0, x0_1])
    fk = f(xk)
    gk_prev = None
    dk = None

    for k in range(max_iter):
        if not np.isfinite(fk) or abs(fk) > 1e8:
            return (i, j, [k, np.nan, xk[0], xk[1], 0])

        gk = gf(xk)
        if not np.all(np.isfinite(gk)):
            return (i, j, [k, np.nan, xk[0], xk[1], 0])
        gn = np.linalg.norm(gk)
        if gn < eps:
            return (i, j, [k, fk, xk[0], xk[1], 1])

        if dk is None:
            dk = -gk
        else:
            gpg = np.dot(gk_prev, gk_prev)
            if gpg < 1e-30:
                dk = -gk
            elif method == "FR":
                dk = -gk + (np.dot(gk, gk) / gpg) * dk
            else:
                dk = -gk + (np.dot(gk, gk - gk_prev) / gpg) * dk

        gd = np.dot(gk, dk)

        eta = 1.0
        for _ in range(50):
            xt = xk + eta * dk
            ft = f(xt)
            if ft <= fk + 0.25 * eta * gd:
                break
            eta *= 0.5

        xk, fk = xt, ft
        gk_prev = gk.copy()

    return (i, j, [max_iter, fk, xk[0], xk[1], 0])

In [ ]:
CACHE_PATH = "../results_b2.npz"


def run_all():
    all_results = {}
    for func_name, cfg in FUNC_CONFIGS.items():
        lo, hi = cfg["domain"]
        xs = np.linspace(lo, hi, GRID)
        ys = np.linspace(lo, hi, GRID)
        for gamma in cfg["gammas"]:
            for method in ("FR", "PR"):
                key = f"{func_name}_g{gamma}_{method}"
                res = np.full((GRID, GRID, 5), np.nan)
                tasks = [
                    (func_name, gamma, method, i, j, xs[j], ys[i], eps, max_iter)
                    for i in range(GRID)
                    for j in range(GRID)
                ]
                for i, j, vals in tqdm(
                    Parallel(n_jobs=-1, return_as="generator")(
                        delayed(_run_one)(*t) for t in tasks
                    ),
                    total=len(tasks),
                    desc=key,
                ):
                    res[i, j] = vals
                all_results[key] = res
    return all_results


if os.path.exists(CACHE_PATH):
    loaded = np.load(CACHE_PATH)
    results = {k: loaded[k] for k in loaded.files}
    print(f"Charge depuis le cache : {len(results)} configurations")
else:
    results = run_all()
    np.savez(CACHE_PATH, **results)
    print("Benchmark termine, resultats sauvegardes.")

In [ ]:
ITER, FVAL, X0, X1, CONV = 0, 1, 2, 3, 4


def global_min_candidates(func_name, gamma):
    """Minima globaux : analytique si connu, sinon evaluation sur grille fine."""
    if func_name == "rosenbrock":
        return [np.array([1.0, 1.0])]

    if func_name == "cubic2":
        disc = 4 * gamma**2 - 12
        if disc < 0:
            return []
        xm = (-2 * gamma + np.sqrt(disc)) / 6
        return [np.array([xm, xm])]

    lo, hi = FUNC_CONFIGS["multitrous2"]["domain"]
    t = np.linspace(lo, hi, 500)
    Xg, Yg = np.meshgrid(t, t)
    Z = 20 * np.cos(Xg**2) + Xg**2 + 20 * np.cos(Yg**2) + gamma * Yg**2
    idx = np.unravel_index(Z.argmin(), Z.shape)
    x0, y0 = abs(Xg[idx]), abs(Yg[idx])
    return [np.array([s0 * x0, s1 * y0]) for s0 in [1, -1] for s1 in [1, -1]]


def classify_basin(func_name, gamma, tol=0.5):
    """0=aucun, 1=FR seul, 2=PR seul, 3=les deux."""
    candidates = global_min_candidates(func_name, gamma)
    if not candidates:
        return np.zeros((GRID, GRID), dtype=int)

    fr = results[f"{func_name}_g{gamma}_FR"]
    pr = results[f"{func_name}_g{gamma}_PR"]

    def near_global(res):
        best = np.full((GRID, GRID), np.inf)
        for c in candidates:
            d = np.sqrt((res[..., X0] - c[0])**2 + (res[..., X1] - c[1])**2)
            best = np.minimum(best, d)
        return (res[..., CONV] > 0.5) & (best < tol)

    fr_ok = near_global(fr)
    pr_ok = near_global(pr)

    cat = np.zeros((GRID, GRID), dtype=int)
    cat[fr_ok & ~pr_ok] = 1
    cat[~fr_ok & pr_ok] = 2
    cat[fr_ok & pr_ok] = 3
    return cat

In [ ]:
print("Resume par configuration :")
print(f"{'Config':<28} {'conv FR':>8} {'conv PR':>8} | "
      f"{'both':>6} {'FR only':>8} {'PR only':>8} {'neither':>8}")
print("-" * 90)
for func_name, cfg in FUNC_CONFIGS.items():
    for gamma in cfg["gammas"]:
        fr = results[f"{func_name}_g{gamma}_FR"]
        pr = results[f"{func_name}_g{gamma}_PR"]
        cat = classify_basin(func_name, gamma)
        fr_conv = fr[..., CONV].mean() * 100
        pr_conv = pr[..., CONV].mean() * 100
        both_pct = (cat == 3).mean() * 100
        fr_only = (cat == 1).mean() * 100
        pr_only = (cat == 2).mean() * 100
        neither = (cat == 0).mean() * 100
        label = f"{func_name} g={gamma}"
        print(f"{label:<28} {fr_conv:>7.1f}% {pr_conv:>7.1f}% | "
              f"{both_pct:>5.1f}% {fr_only:>7.1f}% {pr_only:>7.1f}% {neither:>7.1f}%")
    print()

# Cas convexe : FR = PR sur quadratique

In [ ]:
kappas = [5, 50]
x0_list = [np.array([-3.0, -2.0]), np.array([3.0, -1.0]), np.array([-1.0, 3.5])]
palette = ["#e41a1c", "#377eb8", "#4daf4a"]

fig, axes = plt.subplots(1, len(kappas), figsize=(7 * len(kappas), 6), constrained_layout=True)

for ki, kappa in enumerate(kappas):
    ax = axes[ki]
    A, b = create_system(2, cond=kappa, seed=42)
    x_star = np.linalg.solve(A, b)
    gf_q = lambda x, A=A, b=b: A @ x - b

    pts = np.vstack(x0_list + [x_star])
    lo = pts.min(axis=0) - 1.5
    hi = pts.max(axis=0) + 1.5
    tx = np.linspace(lo[0], hi[0], 200)
    ty = np.linspace(lo[1], hi[1], 200)
    Xg, Yg = np.meshgrid(tx, ty)
    Z = (A[0, 0] * Xg**2 + 2 * A[0, 1] * Xg * Yg + A[1, 1] * Yg**2) / 2 - b[0] * Xg - b[1] * Yg
    ax.contour(Xg, Yg, Z, levels=20, cmap="gray_r", alpha=0.4, linewidths=0.7)

    for xi, x0 in enumerate(x0_list):
        iters = {}
        for method, ls, lw in [("FR", "-", 2.0), ("PR", "--", 1.3)]:
            xk = x0.copy()
            positions = [xk.copy()]
            gk_prev, dk = None, None
            for it in range(max_iter):
                gk = gf_q(xk)
                if np.linalg.norm(gk) < eps:
                    break
                if dk is None:
                    dk = -gk
                else:
                    gpg = np.dot(gk_prev, gk_prev)
                    if gpg < 1e-30:
                        dk = -gk
                    elif method == "FR":
                        dk = -gk + (np.dot(gk, gk) / gpg) * dk
                    else:
                        dk = -gk + (np.dot(gk, gk - gk_prev) / gpg) * dk
                dAd = dk.T @ A @ dk
                if dAd <= 0:
                    break
                eta = -np.dot(gk, dk) / dAd
                xk = xk + eta * dk
                positions.append(xk.copy())
                gk_prev = gk.copy()
            traj = np.array(positions)
            ax.plot(traj[:, 0], traj[:, 1], ls, color=palette[xi], lw=lw, alpha=0.9)
            iters[method] = len(traj) - 1
        ax.plot(*x0, "o", color=palette[xi], ms=7, zorder=5)
        ax.annotate(
            f"FR:{iters['FR']} PR:{iters['PR']}", xy=x0, fontsize=7,
            xytext=(5, 5), textcoords="offset points",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, lw=0),
        )

    ax.plot(*x_star, "k*", ms=14, zorder=10, label=r"$x^\star$")
    ax.set_title(rf"$\kappa={kappa}$")
    ax.set_xlabel(r"$x_1$")
    if ki == 0:
        ax.set_ylabel(r"$x_2$")
        ax.plot([], [], "k-", lw=2.0, label="FR")
        ax.plot([], [], "k--", lw=1.3, label="PR")
        ax.legend(fontsize=9, loc="upper left")

fig.suptitle(
    r"Cas convexe (quadratique, recherche linéaire exacte) : FR $\equiv$ PR",
    fontsize=13, weight="bold",
)
fig.savefig(f"{OUT}/graphique_1.png", dpi=300, bbox_inches="tight")
plt.show()

# Cartes des bassins d'attraction

In [ ]:
BASIN_COLORS = ["#444444", "#1f77b4", "#d62728", "#2ca02c"]
BASIN_LABELS = ["Aucun", "FR seul", "PR seul", "Les deux"]
legend_patches = [Patch(color=c, label=l) for c, l in zip(BASIN_COLORS, BASIN_LABELS)]
basin_cmap = ListedColormap(BASIN_COLORS)
basin_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], basin_cmap.N)
TITLE_MAP = {
    "multitrous2": "Multitrous",
    "cubic2": "Cubique",
    "rosenbrock": "Rosenbrock",
}


def plot_basins(func_name, fig_path):
    gammas = FUNC_CONFIGS[func_name]["gammas"]
    ncols = len(gammas)
    fig, axes = plt.subplots(1, ncols, figsize=(4.5 * ncols, 4.2), constrained_layout=True)
    if ncols == 1:
        axes = [axes]

    lo, hi = FUNC_CONFIGS[func_name]["domain"]
    xs = np.linspace(lo, hi, GRID)
    ys = np.linspace(lo, hi, GRID)

    for gi, gamma in enumerate(gammas):
        ax = axes[gi]
        cat = classify_basin(func_name, gamma)

        fr = results[f"{func_name}_g{gamma}_FR"]
        pr = results[f"{func_name}_g{gamma}_PR"]
        fr_conv = fr[..., CONV].mean() * 100
        pr_conv = pr[..., CONV].mean() * 100
        both_global = (cat == 3).mean() * 100

        ax.pcolormesh(xs, ys, cat, cmap=basin_cmap, norm=basin_norm, shading="nearest")

        ax.set_title(
            rf"$\gamma={gamma}$" + "\n"
            + f"conv FR {fr_conv:.0f}% | PR {pr_conv:.0f}% | glob {both_global:.0f}%",
            fontsize=10,
        )
        ax.set_xlabel(r"$x_0^{(1)}$")
        if gi == 0:
            ax.set_ylabel(r"$x_0^{(2)}$")
        ax.set_aspect("equal")

    fig.suptitle(f"Bassins d'attraction — {TITLE_MAP[func_name]}", fontsize=14, weight="bold")
    fig.legend(handles=legend_patches, loc="lower center", ncol=4, fontsize=10,
               bbox_to_anchor=(0.5, -0.04))
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
plot_basins("multitrous2", f"{OUT}/graphique_2.png")

In [ ]:
plot_basins("cubic2", f"{OUT}/graphique_3.png")

# Distribution du nombre d'itérations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
func_order = ["multitrous2", "cubic2", "rosenbrock"]

for fi, func_name in enumerate(func_order):
    ax = axes[fi]
    gammas = FUNC_CONFIGS[func_name]["gammas"]
    n_g = len(gammas)
    width = 0.35
    pos_fr = np.arange(n_g) * 2.0
    pos_pr = pos_fr + width + 0.05

    data_fr, data_pr = [], []
    for gamma in gammas:
        fr = results[f"{func_name}_g{gamma}_FR"]
        pr = results[f"{func_name}_g{gamma}_PR"]
        fr_it = fr[..., ITER][fr[..., CONV] > 0.5]
        pr_it = pr[..., ITER][pr[..., CONV] > 0.5]
        data_fr.append(fr_it if len(fr_it) > 0 else np.array([np.nan]))
        data_pr.append(pr_it if len(pr_it) > 0 else np.array([np.nan]))

    bp_fr = ax.boxplot(
        data_fr, positions=pos_fr, widths=width,
        patch_artist=True, showfliers=False,
    )
    bp_pr = ax.boxplot(
        data_pr, positions=pos_pr, widths=width,
        patch_artist=True, showfliers=False,
    )
    for p in bp_fr["boxes"]:
        p.set_facecolor("#1f77b4")
        p.set_alpha(0.7)
    for p in bp_pr["boxes"]:
        p.set_facecolor("#d62728")
        p.set_alpha(0.7)

    ax.set_xticks((pos_fr + pos_pr) / 2)
    ax.set_xticklabels([rf"$\gamma={g}$" for g in gammas])
    ax.set_ylabel("Itérations")
    ax.set_title(TITLE_MAP[func_name])
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.3, ls="--")

axes[0].legend([bp_fr["boxes"][0], bp_pr["boxes"][0]], ["FR", "PR"], loc="upper left")
fig.suptitle("Distribution du nombre d'itérations (exécutions convergées)", fontsize=14, weight="bold")
fig.savefig(f"{OUT}/graphique_4.png", dpi=300, bbox_inches="tight")
plt.show()

# Trajectoires emblématiques sur Rosenbrock

In [ ]:
def run_trajectory(func_name, gamma, method, x0):
    """Descente CG complete avec enregistrement de la trajectoire."""
    if func_name == "multitrous2":
        f = lambda x: multitrous2_(x, gamma=gamma)
        gf = lambda x: grad_multitrous2(x, gamma)
    elif func_name == "cubic2":
        f = lambda x: cubic2_(x, gamma=gamma)
        gf = lambda x: grad_cubic2(x, gamma)
    else:
        f = lambda x: Rosenbrock(x, gamma=gamma)
        gf = lambda x: grad_rosenbrock(x, gamma)

    xk = x0.copy()
    fk = f(xk)
    positions = [xk.copy()]
    gk_prev = None
    dk = None

    for k in range(max_iter):
        if not np.isfinite(fk) or abs(fk) > 1e8:
            break
        gk = gf(xk)
        if not np.all(np.isfinite(gk)):
            break
        gn = np.linalg.norm(gk)
        if gn < eps:
            break
        if dk is None:
            dk = -gk
        else:
            gpg = np.dot(gk_prev, gk_prev)
            if gpg < 1e-30:
                dk = -gk
            elif method == "FR":
                dk = -gk + (np.dot(gk, gk) / gpg) * dk
            else:
                dk = -gk + (np.dot(gk, gk - gk_prev) / gpg) * dk

        gd = np.dot(gk, dk)

        eta = 1.0
        for _ in range(50):
            xt = xk + eta * dk
            ft = f(xt)
            if ft <= fk + 0.25 * eta * gd:
                break
            eta *= 0.5

        xk, fk = xt, ft
        positions.append(xk.copy())
        gk_prev = gk.copy()

    return np.array(positions)

In [ ]:
traj_gammas = [1, 10, 100]
n_traj = 3

fig, axes = plt.subplots(
    1, len(traj_gammas), figsize=(6 * len(traj_gammas), 5.5), constrained_layout=True,
)

for gi, gamma in enumerate(traj_gammas):
    ax = axes[gi]

    lo, hi = FUNC_CONFIGS["rosenbrock"]["domain"]
    t = np.linspace(lo, hi, 300)
    Xg, Yg = np.meshgrid(t, t)
    Z = (Xg - 1)**2 + gamma * (Xg**2 - Yg)**2
    ax.contour(Xg, Yg, Z, levels=np.logspace(-1, 3.5, 25), cmap="gray_r", alpha=0.4, linewidths=0.7)

    fr_res = results[f"rosenbrock_g{gamma}_FR"]
    pr_res = results[f"rosenbrock_g{gamma}_PR"]
    xs = np.linspace(lo, hi, GRID)
    ys = np.linspace(lo, hi, GRID)

    both_conv = (fr_res[..., CONV] > 0.5) & (pr_res[..., CONV] > 0.5)
    diff = np.where(both_conv, np.abs(fr_res[..., ITER] - pr_res[..., ITER]), 0)
    flat_idx = np.argsort(diff.ravel())[::-1][:n_traj]

    palette = ["#e41a1c", "#377eb8", "#4daf4a"]

    for ti, fi_flat in enumerate(flat_idx):
        iy, ix = np.unravel_index(fi_flat, (GRID, GRID))
        x0 = np.array([xs[ix], ys[iy]])

        traj_fr = run_trajectory("rosenbrock", gamma, "FR", x0)
        traj_pr = run_trajectory("rosenbrock", gamma, "PR", x0)

        ax.plot(traj_fr[:, 0], traj_fr[:, 1], "-", color=palette[ti], lw=1.2, alpha=0.9)
        ax.plot(traj_pr[:, 0], traj_pr[:, 1], "--", color=palette[ti], lw=1.2, alpha=0.9)
        ax.plot(*x0, "o", color=palette[ti], ms=6, zorder=5)

        it_fr, it_pr = len(traj_fr) - 1, len(traj_pr) - 1
        ax.annotate(
            f"FR:{it_fr} PR:{it_pr}", xy=x0, fontsize=7,
            xytext=(5, 5), textcoords="offset points",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, lw=0),
        )

    ax.plot(1, 1, "k*", ms=12, zorder=10, label=r"$x^\star$")
    ax.set_title(rf"$\gamma={gamma}$")
    ax.set_xlabel(r"$x_1$")
    if gi == 0:
        ax.set_ylabel(r"$x_2$")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

    if gi == 0:
        ax.plot([], [], "k-", lw=1.2, label="FR")
        ax.plot([], [], "k--", lw=1.2, label="PR")
        ax.legend(fontsize=9, loc="upper left")

fig.suptitle(
    "Trajectoires FR (trait plein) et PR (pointillé) sur la fonction de Rosenbrock",
    fontsize=13, weight="bold",
)
fig.savefig(f"{OUT}/graphique_5.png", dpi=300, bbox_inches="tight")
plt.show()